In [1]:
import vame
from vame.util.sample_data import download_sample_data
from pathlib import Path

2026-07-21 20:33:21.92 INFO  --- [MainThread] numexpr.utils   : 151 : Note: NumExpr detected 36 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-07-21 20:33:21.92 INFO  --- [MainThread] numexpr.utils   : 164 : NumExpr defaulting to 16 threads.
2026-07-21 20:33:23.870 INFO  --- [MainThread] vame.util.auxiliary : 43 : Using CUDA — GPU: Quadro RTX 5000
2026-07-21 20:33:24.439 INFO  --- [MainThread] vame.util.auxiliary : 43 : Using CUDA — GPU: Quadro RTX 5000
2026-07-21 20:33:38.440 | WARNING  | tqdm.autonotebook:<module>:28 - d:\Anaconda3\envs\vame\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



In [2]:
pose_files = sorted(
    str(path)
    for path in Path(r"C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\trainingSessions\unifiedDataset_topview_DLC_singleAnimal\analyzedVideos_unifiedDataset_DLC_singleAnimal_downsampled_10fps").rglob("*.h5")
)

config_file, config = vame.init_new_project(
    project_name="unifiedDataset_topView_VAME_Resident",
    poses_estimations=pose_files,
    source_software="DeepLabCut",
    fps=10,
)

2026-07-21 20:34:00.27 INFO  --- [MainThread] vame.initialize_project.new : 116 : Created "C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\neuroPixels_aggressionTraining\unifiedDataset_topView_VAME_Resident\data"
2026-07-21 20:34:00.28 INFO  --- [MainThread] vame.initialize_project.new : 116 : Created "C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\neuroPixels_aggressionTraining\unifiedDataset_topView_VAME_Resident\data\raw"
2026-07-21 20:34:00.29 INFO  --- [MainThread] vame.initialize_project.new : 116 : Created "C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\neuroPixels_aggressionTraining\unifiedDataset_topView_VAME_Resident\data\processed"
2026-07-21 20:34:00.32 INFO  --- [MainThread] vame.initialize_project.new : 116 : Created "C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\neuroPixels_aggressionTraining\unifiedDataset_topView_VAME_Resident\results"
2026-07-21 20:34:00.33 INFO  --- [MainThread] vame.initialize_project.new : 116 : Created "C:\Users\stagk\O

In [5]:
import numpy as np
import pandas as pd
import xarray as xr


def inspect_vame_nans(config: dict) -> pd.DataFrame:
    """Report NaN/Inf values by session, stage, keypoint, and individual."""
    project_path = Path(config["project_path"])
    rows = []

    variables = [
        "position",
        "position_cleaned_lowconf",
        "position_egocentric_aligned",
        "position_processed",
    ]

    for session in config["session_names"]:
        file_path = (
            project_path
            / "data"
            / "processed"
            / f"{session}_processed.nc"
        )

        with xr.open_dataset(file_path) as ds:
            for variable in variables:
                if variable not in ds:
                    continue

                data = ds[variable]

                for keypoint in data.coords["keypoints"].values:
                    selected = data.sel(keypoints=keypoint)

                    if "individuals" in selected.dims:
                        individuals = selected.coords["individuals"].values
                    else:
                        individuals = [None]

                    for individual in individuals:
                        series = (
                            selected.sel(individuals=individual)
                            if individual is not None
                            else selected
                        )

                        values = np.asarray(series.values)
                        nonfinite = ~np.isfinite(values)

                        rows.append(
                            {
                                "session": session,
                                "variable": variable,
                                "keypoint": str(keypoint),
                                "individual": (
                                    str(individual)
                                    if individual is not None
                                    else "single"
                                ),
                                "nonfinite_values": int(nonfinite.sum()),
                                "total_values": int(values.size),
                                "nonfinite_percent": (
                                    100.0 * nonfinite.mean()
                                ),
                                "all_nonfinite": bool(nonfinite.all()),
                            }
                        )

    report = pd.DataFrame(rows)
    return report.sort_values(
        ["nonfinite_percent", "session"],
        ascending=[False, True],
    )


nan_report = inspect_vame_nans(config)

display(
    nan_report.loc[
        nan_report["nonfinite_values"] > 0
    ]
)

,session,variable,keypoint,individual,nonfinite_values,total_values,nonfinite_percent,all_nonfinite
9371,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,position_cleaned_lowconf,rightEar,individual_0,2570,2570,100.000000,True
9372,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,position_cleaned_lowconf,leftEar,individual_0,2570,2570,100.000000,True
9379,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,position_cleaned_lowconf,tailTip,individual_0,2570,2570,100.000000,True
9381,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,position_egocentric_aligned,rightEar,individual_0,2570,2570,100.000000,True
9382,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,position_egocentric_aligned,leftEar,individual_0,2570,2570,100.000000,True
...,...,...,...,...,...,...,...,...
29005,mouse975827_Day06_topView_compDLC_Resnet101_tr...,position,midPoint,individual_0,2,25336,0.007894,False
2685,mouse1029736_Day09_topView_compDLC_Resnet101_t...,position,midPoint,individual_0,2,25806,0.007750,False
1925,mouse1010832_Day16_topView_compDLC_Resnet101_t...,position,midPoint,individual_0,2,26002,0.007692,False
31644,mouse978775_Day04_topView_compDLC_Resnet101_tr...,position,betNeckBaseAndMidPoint,individual_0,2,26636,0.007509,False


In [6]:
def inspect_vame_confidence(config: dict) -> pd.DataFrame:
    project_path = Path(config["project_path"])
    threshold = float(config["pose_confidence"])
    rows = []

    for session in config["session_names"]:
        file_path = (
            project_path
            / "data"
            / "processed"
            / f"{session}_processed.nc"
        )

        with xr.open_dataset(file_path) as ds:
            confidence = ds["confidence"]

            for keypoint in confidence.coords["keypoints"].values:
                selected = confidence.sel(keypoints=keypoint)

                if "individuals" in selected.dims:
                    individuals = selected.coords["individuals"].values
                else:
                    individuals = [None]

                for individual in individuals:
                    values = np.asarray(
                        selected.sel(individuals=individual).values
                        if individual is not None
                        else selected.values
                    )

                    invalid = (
                        ~np.isfinite(values)
                        | (values < threshold)
                    )

                    rows.append(
                        {
                            "session": session,
                            "keypoint": str(keypoint),
                            "individual": (
                                str(individual)
                                if individual is not None
                                else "single"
                            ),
                            "median_confidence": float(
                                np.nanmedian(values)
                            ),
                            "minimum_confidence": float(
                                np.nanmin(values)
                            ),
                            "percent_below_threshold": float(
                                100.0 * invalid.mean()
                            ),
                            "all_below_threshold": bool(
                                invalid.all()
                            ),
                        }
                    )

    return pd.DataFrame(rows).sort_values(
        "percent_below_threshold",
        ascending=False,
    )


confidence_report = inspect_vame_confidence(config)
print("VAME pose-confidence threshold:", config["pose_confidence"])

display(confidence_report.head(30))

VAME pose-confidence threshold: 0.99


,session,keypoint,individual,median_confidence,minimum_confidence,percent_below_threshold,all_below_threshold
2342,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,leftEar,individual_0,0.360183,0.014563,100.000000,True
2341,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,rightEar,individual_0,0.345186,0.007168,100.000000,True
2349,mouse1079906_Day001_pt1DLC_Resnet101_unifiedDa...,tailTip,individual_0,0.061442,0.004782,100.000000,True
2849,mouse1087824_Day001DLC_Resnet101_unifiedDatase...,tailTip,individual_0,0.128564,0.006179,99.969697,False
2902,mouse1087825_Day001DLC_Resnet101_unifiedDatase...,leftEar,individual_0,0.250186,0.006674,99.962121,False
2901,mouse1087825_Day001DLC_Resnet101_unifiedDatase...,rightEar,individual_0,0.243929,0.006572,99.946970,False
2359,mouse1079906_Day001_pt2DLC_Resnet101_unifiedDa...,tailTip,individual_0,0.152408,0.003398,99.946154,False
2842,mouse1087824_Day001DLC_Resnet101_unifiedDatase...,leftEar,individual_0,0.602320,0.008737,99.931818,False
2909,mouse1087825_Day001DLC_Resnet101_unifiedDatase...,tailTip,individual_0,0.123639,0.003130,99.924242,False
2682,mouse1087821_Day001_pt1DLC_Resnet101_unifiedDa...,leftEar,individual_0,0.579479,0.014523,99.922179,False


In [7]:
print(config["pose_confidence"])

0.99


In [8]:
config["pose_confidence"] = 0.5

In [9]:
vame.preprocessing(
    config=config,
    centered_reference_keypoint="midPoint",
    orientation_reference_keypoint="tailBase",
)

2026-07-21 20:45:45.304 INFO  --- [MainThread] vame.preprocessing.preprocessing : 67 : Cleaning low confidence data points...
2026-07-21 20:45:45.306 INFO  --- [MainThread] vame.preprocessing.cleaning : 46 : Cleaning low confidence data points. Confidence threshold: 0.5
2026-07-21 20:45:45.306 INFO  --- [MainThread] vame.preprocessing.cleaning : 49 : Session: mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk
2026-07-21 20:45:45.362 INFO  --- [MainThread] vame.preprocessing.cleaning : 49 : Session: mouse1010819_Day04_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk
2026-07-21 20:45:45.403 INFO  --- [MainThread] vame.preprocessing.cleaning : 49 : Session: mouse1010819_Day05_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk
2026-07-21 20:45:45.456 INFO  --- [MainThread] vame.preprocessing.cleaning : 49 : Session:

'position_processed'

In [10]:
vame.create_trainset(
    config=config,
    test_fraction=0.1,
    split_mode="mode_2",
)

2026-07-21 20:49:50.834 INFO  --- [MainThread] vame.model.create_training : 320 : Creating training dataset...
2026-07-21 20:50:11.795 INFO  --- [MainThread] vame.model.create_training : 187 : Session mouse1010819_Day02_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk: test chunk 7270:8543 (length 1273)
2026-07-21 20:50:11.795 INFO  --- [MainThread] vame.model.create_training : 187 : Session mouse1010819_Day04_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk: test chunk 860:2082 (length 1222)
2026-07-21 20:50:11.815 INFO  --- [MainThread] vame.model.create_training : 187 : Session mouse1010819_Day05_topView_compDLC_Resnet101_trainingAggression_trainingSessions_topViewFeb3shuffle2_snapshot_best-100_sk: test chunk 5390:6599 (length 1209)
2026-07-21 20:50:11.816 INFO  --- [MainThread] vame.model.create_training : 187 : Session mouse1010819_Day06_topView_compDLC_Resnet101_trainingAgg

In [11]:
vame.train_model(config=config)

2026-07-21 20:50:14.611 INFO  --- [MainThread] vame.model.rnn_vae : 559 : Train Variational Autoencoder - model name: VAME 

2026-07-21 20:50:14.619 INFO  --- [MainThread] vame.model.rnn_vae : 575 : TensorBoard logging enabled. Log directory: C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\neuroPixels_aggressionTraining\unifiedDataset_topView_VAME_Resident\logs\tensorboard\VAME
2026-07-21 20:50:14.620 INFO  --- [MainThread] vame.model.rnn_vae : 576 : To view logs, run: tensorboard --logdir=C:\Users\stagk\OneDrive\Desktop\trainingAggression_MR\neuroPixels_aggressionTraining\unifiedDataset_topView_VAME_Resident\logs\tensorboard --port=6006
2026-07-21 20:50:14.622 INFO  --- [MainThread] vame.util.auxiliary : 43 : Using CUDA — GPU: Quadro RTX 5000
2026-07-21 20:50:14.622 INFO  --- [MainThread] vame.model.rnn_vae : 619 : Latent Dimensions: 30, Time window: 30, Batch Size: 256, Beta: 1, lr: 0.0005

2026-07-21 20:50:15.295 INFO  --- [MainThread] vame.model.rnn_vae : 655 : Model graph lo

KeyboardInterrupt: 

In [ ]:
vame.evaluate_model(config=config)